# 05.02 — Mini Project: Pilihan Mu

**Tujuan**: closing yang aktif — pilih **satu task NLP Bahasa Indonesia** yang kamu peduli, ship dengan tool yang paling cocok berdasarkan framework modul 05.01.

**Output**: 1 notebook end-to-end yang bisa kamu pajang di portfolio GitHub.

**Prasyarat**: modul 05.01 lulus.

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Briefing — apa yang harus kamu lakukan

Tahap-tahap:

### Step 1: Pilih task (5 menit)

Pilih **satu** dari kategori berikut, atau bikin sendiri:

- **Klasifikasi**: deteksi spam SMS Bahasa, klasifikasi topik berita, deteksi hate speech, sentiment review e-commerce.
- **Extraction**: ekstrak nama orang/lokasi dari berita, ekstrak harga + nama barang dari iklan, parsing alamat.
- **Generation**: summarize laporan keuangan, rewrite artikel formal → casual, jawab FAQ dari knowledge base.
- **QA / RAG**: jawab pertanyaan dari dokumen internal mu sendiri (mis. catatan kuliah, dokumentasi project).

### Step 2: Isi template di section 2 (10 menit)

Jawab 5 pertanyaan decision framework untuk task kamu, lalu pakai `recommend()` dari 05.01 untuk dapat rekomendasi.

### Step 3: Implement minimal viable version (30-60 menit)

Pakai pattern dari notebook yang relevan:
- Klasifikasi → adaptasi 03.01 (fine-tune DistilBERT)
- Extraction → adaptasi 04.01 (Groq few-shot) atau 03.01 (fine-tune NER)
- Generation → adaptasi 04.02 (Groq summarize)
- QA / RAG → adaptasi 04.03 (TF-IDF + Groq)

### Step 4: Tulis refleksi (5 menit)

Apa yang kamu pelajari? Hasil sesuai ekspektasi? Apa yang akan kamu coba next?

## 2. Template: rencana proyek mu

**Tugas**: edit cell di bawah, ganti placeholder dengan jawaban kamu sendiri.

### 📋 Project: [GANTI dengan nama project kamu]

**Apa yang ingin kamu bangun, dalam 1 kalimat?**

> _Contoh: "Klasifikasi otomatis review produk Tokopedia ke 5 kelas: sangat positif, positif, netral, negatif, sangat negatif."_

[ISI: ...]

**Siapa user akhirnya? Use case nya apa?**

[ISI: ...]

**Data yang akan kamu pakai (sumber, ukuran)?**

[ISI: ...]

**Cara evaluasi success?**

> _Contoh: "accuracy ≥ 80% di test set 200 review, latency ≤ 200ms/predict."_

[ISI: ...]

## 3. Jalankan decision framework

In [ ]:
from dataclasses import dataclass

@dataclass
class ProjectSpec:
    name: str
    data_sensitive: bool
    needs_offline: bool
    complex_reasoning: bool
    narrow_task: bool
    has_labeled_data: bool
    requests_per_day: int
    latency_p95_ms: int

def recommend(spec: ProjectSpec) -> dict:
    if spec.needs_offline:
        return {"choice": "SLM lokal (quantized)", "reason": "offline → no API"}
    if spec.data_sensitive:
        if spec.narrow_task and spec.has_labeled_data:
            return {"choice": "SLM fine-tuned", "reason": "privacy + narrow + data → fine-tune wins"}
        return {"choice": "SLM lokal general", "reason": "privacy hard requirement"}
    if spec.complex_reasoning:
        return {"choice": "LLM API besar (70b+)", "reason": "reasoning butuh kapasitas"}
    if spec.narrow_task and spec.has_labeled_data and spec.requests_per_day > 10_000:
        return {"choice": "SLM fine-tuned", "reason": "narrow + data + volume → fine-tune jauh lebih murah"}
    if spec.latency_p95_ms < 500:
        return {"choice": "LLM API (Groq) atau SLM lokal", "reason": "latency ketat"}
    return {"choice": "LLM API default (Groq llama-3.1-8b)", "reason": "start simple"}


# GANTI nilai di bawah sesuai project mu
my_project = ProjectSpec(
    name="My Project Name",            # ganti
    data_sensitive=False,                # ganti True/False
    needs_offline=False,                 # ganti
    complex_reasoning=False,             # ganti
    narrow_task=True,                    # ganti
    has_labeled_data=True,               # ganti
    requests_per_day=1_000,              # ganti
    latency_p95_ms=2_000,                # ganti
)

rec = recommend(my_project)
print(f"📌 {my_project.name}")
print(f"   → {rec['choice']}")
print(f"   karena: {rec['reason']}")

## 4. Implementasi (skeleton — adaptasi sesuai task)

Code di bawah adalah **starter** untuk 4 jenis task. Pilih yang sesuai rekomendasi di section 3, lalu adaptasi.

### Opsi A — LLM API (Groq few-shot)

Cocok kalau: rekomendasi nya "LLM API" + task kamu generative / klasifikasi tanpa labeled data.

In [ ]:
# UNCOMMENT & adaptasi kalau pilihanmu Opsi A

# from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client
# client = groq_client()

# SYSTEM = "Kamu adalah ..."  # define role model
# FEW_SHOT = """..."""          # 3-5 contoh input → output

# def predict(text):
#     r = client.chat.completions.create(
#         model=GROQ_DEFAULT_MODEL,
#         messages=[
#             {"role": "system", "content": SYSTEM},
#             {"role": "user", "content": f"{FEW_SHOT}\n\nInput: {text}\nOutput:"},
#         ],
#         temperature=0, max_tokens=50,
#     )
#     return r.choices[0].message.content.strip()

# # Test
# print(predict("YOUR TEST INPUT"))
print("Lihat 04.01 untuk contoh lengkap.")

### Opsi B — Fine-tune SLM (DistilBERT)

Cocok kalau: rekomendasi nya "SLM fine-tuned" + task klasifikasi/extraction + ada labeled data.

In [ ]:
# UNCOMMENT & adaptasi kalau pilihanmu Opsi B
# Pattern lengkap di notebook 03.01

# from datasets import load_dataset, Dataset
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# # 1. Load data mu (CSV, JSON, atau HF dataset)
# # df = pd.read_csv("data/raw/my_data.csv")
# # ds = Dataset.from_pandas(df)

# # 2. Tokenize, train Trainer, evaluate — copy paste pattern dari 03.01
print("Lihat 03.01 untuk fine-tune pipeline lengkap.")

### Opsi C — Mini RAG

Cocok kalau: kamu punya knowledge base + butuh QA / generation berbasis konteks.

In [ ]:
# UNCOMMENT & adaptasi kalau pilihanmu Opsi C
# Pattern lengkap di notebook 04.03

# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# from utils.llm_clients import groq_client, GROQ_DEFAULT_MODEL

# # 1. Build knowledge base (chunks)
# # 2. Vectorize
# # 3. Retrieve top-k for question
# # 4. Prompt LLM dengan context + question
print("Lihat 04.03 untuk RAG pipeline lengkap.")

### Opsi D — SLM lokal quantized (offline)

Cocok kalau: privacy / offline hard requirement.

In [ ]:
# UNCOMMENT & adaptasi kalau pilihanmu Opsi D
# Pattern lengkap di notebook 02.03

# from huggingface_hub import hf_hub_download
# from llama_cpp import Llama

# # 1. Download GGUF (TinyLlama / Phi-3-mini / SmolLM2-1.7B Q4)
# # 2. Load via llama_cpp.Llama(...)
# # 3. Pakai .create_chat_completion(messages=...) — sama persis pattern Groq
print("Lihat 02.03 untuk pattern llama-cpp lengkap.")

## 5. Checklist sebelum kamu commit project ini ke GitHub

- [ ] Notebook ini di-rename jadi nama yang representatif (mis. `02_proyek_sentiment_tokopedia.ipynb`).
- [ ] Section 2 (template plan) terisi lengkap.
- [ ] Code di section 4 sudah dijalankan & ada output sample.
- [ ] Ada **min 3 test case** dengan input → output beneran.
- [ ] Ada **metrik evaluasi** (accuracy, latency, dll) sesuai "success criteria" di section 2.
- [ ] Refleksi di section 6 ditulis.
- [ ] Data sensitif (kalau ada) **tidak ikut di-commit** (cek `.gitignore`).
- [ ] README utama repo di-update: tambahkan section project mu di bagian "Mini Projects".

## 6. Refleksi (isi setelah implementasi)

**Apa hasil utama project ini?**

[ISI: angka metrik + insight kunci]

**Apakah rekomendasi decision framework cocok dengan hasil?** Kalau tidak, kenapa?

[ISI: ...]

**Apa yang akan kamu coba next iteration?**

[ISI: ...]

**Bagaimana cara kamu *production-ize* ini kalau ada user real?**

[ISI: ...]

## Selesai

Kalau notebook ini sudah terisi dan jalan, kamu sudah **menyelesaikan seluruh kurikulum `llm-vs-slm-lab`**.

Yang kamu kuasai sekarang:

1. Beda LLM vs SLM konseptual & praktis (modul 01)
2. Pattern inference LLM API (Groq) & SLM lokal (transformers + llama-cpp) (modul 02)
3. Fine-tune model encoder kecil di CPU (modul 03)
4. Bangun studi kasus head-to-head dengan metrik & visualisasi (modul 04)
5. Pilih model dengan justifikasi terstruktur untuk proyek nyata (modul 05)

Lanjut belajar:
- Internals LLM mendalam → [`llm-internals`](../../llm-internals/)
- Neural network dari nol (autograd, mini-GPT) → [`neural-from-scratch`](../../neural-from-scratch/)
- MLOps untuk production → `mlops-in-production` (di workspace yang sama)

Selamat — kamu sekarang punya **mental model + portfolio konkret** untuk diskusi LLM vs SLM di proyek tim mu.